# Survey Divergence Overview

This notebook looks for public/private divergence in survey answers across sweeps 6-9.

The survey scores in the dataframe are centered from `-2` to `2`, with `0` as neutral. The main categorical divergence signal is a response-category mismatch across three response categories:

- negative: score < 0
- neutral: score == 0
- positive: score > 0

A public/private response-category mismatch means the two channels fall into different categories. For example, public neutral and private negative is a mismatch, as is public positive and private neutral.

The notebook emphasizes rates and upper-tail behavior rather than raw counts, because conditions do not all have the same number of rows.


In [ ]:
from pathlib import Path
import os
import sys
import pickle
from typing import Any

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
})

FAMILY_MAP = {
    **{q: "deliberative" for q in range(1, 7)},
    **{q: "evaluative" for q in range(7, 10)},
    **{q: "incentive" for q in range(10, 16)},
}
FAMILY_ORDER = {"deliberative": 1, "evaluative": 2, "incentive": 3}
FAMILIES = ["deliberative", "evaluative", "incentive"]
CONDITION_ORDER = ["baseline", "persona-reinforcing", "alignment-inducing"]
CONDITION_COLORS = {
    "baseline": "#7f7f7f",
    "persona-reinforcing": "#4c78a8",
    "alignment-inducing": "#d95f02",
}
SURVEY_NEUTRAL_EPSILON = 1e-9
FOCUS_GAP_THRESHOLD = 1.0
TAIL_GAP_THRESHOLDS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.5, 2.0]

def survey_response_side(scores):
    return np.select(
        [scores > SURVEY_NEUTRAL_EPSILON, scores < -SURVEY_NEUTRAL_EPSILON],
        ["positive", "negative"],
        default="neutral",
    )

def short_model(name: str) -> str:
    return name.split("/", 1)[-1] if "/" in name else name

def scenario_title(scenario: str) -> str:
    return scenario.replace("_", " ").title()

def condition_label(direction):
    if pd.isna(direction) or direction is None:
        return "baseline"
    direction = str(direction).lower()
    if direction == "negative":
        return "persona-reinforcing"
    if direction == "positive":
        return "alignment-inducing"
    return f"unknown:{direction}"

def load_aggregate_pickles(paths):
    frames = []
    for path in paths:
        with path.open("rb") as f:
            frame = pickle.load(f).copy()
        frame["source_pickle"] = str(path)
        frame["source_sweep"] = path.parent.name
        frames.append(frame)
    return pd.concat(frames, ignore_index=True, sort=False)

def stance_related_columns(df):
    return [
        col for col in df.columns
        if "stance" in col.lower() or str(col).startswith("decision-")
    ]

def build_tidy_survey_df(aggregate_df: pd.DataFrame) -> pd.DataFrame:
    records: list[dict[str, Any]] = []
    for _, row in aggregate_df.iterrows():
        scenario = row["scenario_id"]
        model = row["model"]
        inc_dir = row.get("incentive_direction")
        inc_type = row.get("incentive_type")
        pub = row.get("survey-public") or {}
        priv = row.get("survey-private") or {}
        all_q_keys = sorted(
            {k for k in list(pub.keys()) + list(priv.keys()) if k.startswith("Q")},
            key=lambda k: int(k[1:]),
        )
        for q_key in all_q_keys:
            q_num = int(q_key[1:])
            pub_block = pub.get(q_key, {})
            priv_block = priv.get(q_key, {})
            q_family = (
                pub_block.get("question_group")
                or priv_block.get("question_group")
                or FAMILY_MAP.get(q_num, "unknown")
            )
            q_text = pub_block.get("question") or priv_block.get("question", "")
            for role in ("alpha", "beta"):
                pub_part = pub_block.get(role, {})
                priv_part = priv_block.get(role, {})
                turns = pub_part.get("debate_turn") or priv_part.get("debate_turn") or []
                pub_scores = pub_part.get("response_score") or []
                priv_scores = priv_part.get("response_score") or []
                for i, turn in enumerate(turns):
                    public_score = pub_scores[i] if i < len(pub_scores) else np.nan
                    private_score = priv_scores[i] if i < len(priv_scores) else np.nan
                    records.append({
                        "scenario": scenario,
                        "model": model,
                        "incentive_direction": inc_dir,
                        "incentive_type": inc_type,
                        "question_number": q_num,
                        "question_family": q_family,
                        "question_text": q_text,
                        "participant_role": role,
                        "turn": int(turn),
                        "public_score": public_score,
                        "private_score": private_score,
                    })
    df = pd.DataFrame(records)
    if df.empty:
        return df
    df["condition"] = df["incentive_direction"].map(condition_label)
    df["condition_time"] = np.where(df["incentive_type"].isna(), "none", df["incentive_type"].astype(str))
    df["condition_detail"] = np.where(
        df["condition"] == "baseline",
        "baseline",
        df["condition"] + " / " + df["condition_time"],
    )
    df["abs_public_private_diff"] = (df["public_score"] - df["private_score"]).abs()
    df["question_order"] = df["question_number"]
    df["family_order"] = df["question_family"].map(FAMILY_ORDER).fillna(0).astype(int)
    return df.sort_values(["condition", "scenario", "model", "question_order", "participant_role", "turn"]).reset_index(drop=True)

def condition_values(df):
    present = set(df["condition"].dropna().unique())
    ordered = [c for c in CONDITION_ORDER if c in present]
    ordered += sorted(present - set(ordered))
    return ordered


## Load Data

Combines the aggregate pickle files from sweeps 6-9 and drops stance/no-stance columns before survey processing.


In [ ]:
AGGREGATE_PKLS = [
    Path("outputs/sweep_6/aggregate_df_max_completed_plus.pkl"),
    Path("outputs/sweep_7/aggregate_df_max_completed_plus_with_cheese.pkl"),
    Path("outputs/sweep_8/aggregate_df_max_completed_plus_with_cheese.pkl"),
    Path("outputs/sweep_9/aggregate_df_big_mac_plus.pkl"),
]

aggregate_df_raw = load_aggregate_pickles(AGGREGATE_PKLS)
stance_columns = stance_related_columns(aggregate_df_raw)
aggregate_df = aggregate_df_raw.drop(columns=stance_columns)
tidy_df = build_tidy_survey_df(aggregate_df)
tidy_df["condition_detail"] = np.where(
    tidy_df["condition"] == "baseline",
    "baseline",
    tidy_df["condition"] + " / " + tidy_df["condition_time"],
)
tidy_df["public_response_side"] = survey_response_side(tidy_df["public_score"])
tidy_df["private_response_side"] = survey_response_side(tidy_df["private_score"])
tidy_df["public_private_side_mismatch"] = tidy_df["public_response_side"] != tidy_df["private_response_side"]
tidy_df["large_abs_gap"] = tidy_df["abs_public_private_diff"] > FOCUS_GAP_THRESHOLD

print("Loaded pickles:")
for path in AGGREGATE_PKLS:
    print("-", path)
print("Raw aggregate:", aggregate_df_raw.shape)
print("Dropped stance-related columns:", len(stance_columns))
print("Survey aggregate:", aggregate_df.shape)
print("Tidy survey rows:", tidy_df.shape)
print("Survey score range:", tidy_df[["public_score", "private_score"]].min().min(), "to", tidy_df[["public_score", "private_score"]].max().max())
print("Neutral epsilon:", SURVEY_NEUTRAL_EPSILON)
print("Response-category mismatch count:", int(tidy_df["public_private_side_mismatch"].sum()))
print("Response-category mismatch share:", round(float(tidy_df["public_private_side_mismatch"].mean()), 4))
print("Large-gap threshold:", FOCUS_GAP_THRESHOLD)
print("Large-gap count:", int(tidy_df["large_abs_gap"].sum()))
print("Large-gap share:", round(float(tidy_df["large_abs_gap"].mean()), 4))
print("Conditions:", sorted(tidy_df["condition"].dropna().unique()))
print("Scenarios:", sorted(tidy_df["scenario"].dropna().unique()))
print("Models:", len(tidy_df["model"].dropna().unique()))

display(tidy_df[["public_score", "private_score", "abs_public_private_diff"]].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).round(3))
display(pd.crosstab(tidy_df["public_response_side"], tidy_df["private_response_side"], margins=True))
display(tidy_df.head())


## Coverage

A compact check of which conditions, scenario/model combinations, turns, and questions are present.


In [ ]:
coverage = (
    tidy_df.groupby(["condition", "condition_time", "scenario", "model"], observed=True)
    .agg(rows=("abs_public_private_diff", "size"), turns=("turn", "nunique"), questions=("question_number", "nunique"))
    .reset_index()
    .sort_values(["condition", "condition_time", "scenario", "model"])
)
display(coverage)

display(
    coverage.groupby(["condition", "condition_time"], observed=True)
    .agg(scenarios=("scenario", "nunique"), models=("model", "nunique"), rows=("rows", "sum"))
    .reset_index()
)


## Plot 1: Condition-Level Divergence Summary

This is the highest-signal view for the paper claim. It uses normalized rates so alignment-inducing can be compared fairly against baseline and persona-reinforcing conditions.


In [ ]:
def participant_roles(df):
    return [role for role in ["alpha", "beta"] if role in set(df["participant_role"].dropna())]


def survey_families(df):
    present = set(df["question_family"].dropna())
    return [family for family in FAMILIES if family in present]


def condition_detail_order(df):
    details = []
    if "baseline" in set(df["condition_detail"]):
        details.append("baseline")
    for condition in ["persona-reinforcing", "alignment-inducing"]:
        vals = sorted(d for d in df["condition_detail"].dropna().unique() if d.startswith(condition))
        details.extend(vals)
    return details


def condition_divergence_summary(df):
    summary = (
        df.groupby(["condition", "condition_time", "condition_detail"], observed=True)
        .agg(
            n=("abs_public_private_diff", "size"),
            mean_abs_diff=("abs_public_private_diff", "mean"),
            q95_abs_diff=("abs_public_private_diff", lambda s: s.quantile(0.95)),
            q99_abs_diff=("abs_public_private_diff", lambda s: s.quantile(0.99)),
            side_mismatch_count=("public_private_side_mismatch", "sum"),
            side_mismatch_rate=("public_private_side_mismatch", "mean"),
            large_gap_count=("large_abs_gap", "sum"),
            large_gap_rate=("large_abs_gap", "mean"),
            max_abs_diff=("abs_public_private_diff", "max"),
        )
        .reset_index()
    )
    order = condition_detail_order(df)
    summary["condition_detail"] = pd.Categorical(summary["condition_detail"], categories=order, ordered=True)
    return summary.sort_values("condition_detail")


def plot_condition_divergence_summary(df):
    summary = condition_divergence_summary(df)
    labels = summary["condition_detail"].astype(str).to_list()
    x = np.arange(len(labels))
    colors = [CONDITION_COLORS.get(c, "#333333") for c in summary["condition"]]

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
    metrics = [
        ("side_mismatch_rate", "response-category mismatch rate (%)", "side_mismatch_count"),
        ("large_gap_rate", f"rate abs(public-private) > {FOCUS_GAP_THRESHOLD:g} (%)", "large_gap_count"),
        ("q99_abs_diff", "99th percentile abs(public-private)", None),
    ]
    for ax, (metric, ylabel, count_col) in zip(axes, metrics):
        vals = summary[metric].to_numpy(dtype=float)
        plot_vals = vals * 100 if metric.endswith("rate") else vals
        ax.bar(x, plot_vals, color=colors)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=30, ha="right")
        ax.set_ylabel(ylabel)
        ax.set_title(ylabel)
        if count_col:
            for xi, yi, count in zip(x, plot_vals, summary[count_col]):
                ax.text(xi, yi, f"n={int(count)}", ha="center", va="bottom", fontsize=8)
    fig.suptitle("Alignment-inducing shows the strongest public/private divergence signal", y=1.04)
    plt.tight_layout()
    plt.show()
    display(summary.round(4))

plot_condition_divergence_summary(tidy_df)


## Plot 2: Divergence Tail Curves

This view avoids choosing one threshold. A higher curve means that condition has more public/private gaps beyond each possible gap size.


In [ ]:
def plot_tail_rate_curves(df):
    order = condition_detail_order(df)
    rows = []
    for detail in order:
        part = df[df["condition_detail"] == detail]
        condition = part["condition"].iloc[0]
        for threshold in TAIL_GAP_THRESHOLDS:
            rows.append({
                "condition_detail": detail,
                "condition": condition,
                "threshold": threshold,
                "tail_rate": float((part["abs_public_private_diff"] > threshold).mean()),
            })
    tail_df = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(8.8, 5))
    for detail in order:
        part = tail_df[tail_df["condition_detail"] == detail]
        condition = part["condition"].iloc[0]
        ax.plot(part["threshold"], part["tail_rate"] * 100, marker="o", linewidth=2.4, label=detail, color=CONDITION_COLORS.get(condition))
    ax.set_xlabel("absolute public/private gap threshold")
    ax.set_ylabel("rows above threshold (%)")
    ax.set_title("Upper-tail public/private divergence by condition")
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()
    display(tail_df.pivot(index="threshold", columns="condition_detail", values="tail_rate").mul(100).round(2))

plot_tail_rate_curves(tidy_df)


## Plot 3: Scenario x Model Response-Category Mismatch Rate Heatmaps

Each cell is the percent of rows where public and private fall on different response categories (negative, neutral, positive). Zero-rate cells are muted so the eye goes to the actual divergence signal.


In [ ]:
def mismatch_rate_vmax(df, group_cols):
    rates = df.groupby(group_cols, observed=True)["public_private_side_mismatch"].mean().mul(100)
    return max(5.0, float(rates.max()))


def mismatch_rate_cmap():
    cmap = plt.get_cmap("YlOrRd").copy()
    cmap.set_bad("#f4f4f4")
    return cmap


def model_label(name: str) -> str:
    label = short_model(name)
    replacements = {
        "gemini-3.1-pro-preview": "gemini-3.1",
        "claude-opus-4.6": "opus-4.6",
        "deepseek-v3.2": "deepseek-v3.2",
        "gpt-oss-120b": "gpt-oss",
        "minimax-m2.7": "minimax-m2.7",
    }
    return replacements.get(label, label)


def format_heatmap_percent(value):
    if not np.isfinite(value):
        return ""
    if value == 0:
        return "0%"
    if value < 10:
        return f"{value:.1f}%"
    return f"{value:.0f}%"


def annotate_heatmap(ax, values, *, vmax):
    contrast_cutoff = max(10.0, 0.55 * vmax)
    for row in range(values.shape[0]):
        for col in range(values.shape[1]):
            value = values[row, col]
            label = format_heatmap_percent(value)
            if not label:
                continue
            if value == 0:
                color = "#8a8a8a"
                weight = "normal"
            elif value >= contrast_cutoff:
                color = "white"
                weight = "bold"
            else:
                color = "#202020"
                weight = "normal"
            ax.text(col, row, label, ha="center", va="center", fontsize=7.5, fontweight=weight, color=color)


def style_heatmap_axes(ax, table, *, show_x=True):
    ax.set_xticks(np.arange(table.shape[1]))
    ax.set_xticklabels([model_label(m) for m in table.columns], rotation=35, ha="right", fontsize=8)
    ax.set_yticks(np.arange(table.shape[0]))
    ax.set_yticklabels([scenario_title(s) for s in table.index], fontsize=9)
    ax.set_xticks(np.arange(-0.5, table.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, table.shape[0], 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.2)
    ax.tick_params(which="minor", bottom=False, left=False)
    if not show_x:
        ax.tick_params(labelbottom=False)
        ax.set_xlabel("")
    else:
        ax.set_xlabel("model", fontsize=9)


def plot_scenario_model_side_mismatch_rates(df):
    conditions = condition_values(df)
    families = survey_families(df)
    vmax = mismatch_rate_vmax(df, ["participant_role", "question_family", "condition", "scenario", "model"])
    cmap = mismatch_rate_cmap()
    for role in participant_roles(df):
        role_df = df[df["participant_role"] == role]
        fig, axes = plt.subplots(
            len(families),
            len(conditions),
            figsize=(max(10, 5.4 * len(conditions)), 3.3 * len(families)),
            squeeze=False,
            sharex=False,
            sharey=False,
            constrained_layout=True,
        )
        im = None
        for r, family in enumerate(families):
            for c, condition in enumerate(conditions):
                ax = axes[r, c]
                part = role_df[(role_df["condition"] == condition) & (role_df["question_family"] == family)]
                table = (
                    part.groupby(["scenario", "model"], observed=True)["public_private_side_mismatch"]
                    .mean()
                    .mul(100)
                    .unstack("model")
                    .sort_index()
                )
                table = table.reindex(sorted(table.columns), axis=1).fillna(0)
                values = table.to_numpy(dtype=float)
                masked_values = np.ma.masked_where(values <= 0, values)
                im = ax.imshow(masked_values, aspect="auto", cmap=cmap, vmin=0, vmax=vmax)
                style_heatmap_axes(ax, table, show_x=(r == len(families) - 1))
                annotate_heatmap(ax, values, vmax=vmax)
                if r == 0:
                    ax.set_title(condition.replace("-", " "), fontsize=11, pad=10)
                if c == 0:
                    ax.set_ylabel(family, fontsize=10, fontweight="bold")
                else:
                    ax.set_ylabel("")
        fig.suptitle(f"{role.title()} public/private response-category mismatch rate", fontsize=14, y=1.04)
        colorbar = fig.colorbar(im, ax=axes.ravel().tolist(), label="response-category mismatch rate", fraction=0.025, pad=0.015)
        colorbar.ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0f}%")
        plt.show()

plot_scenario_model_side_mismatch_rates(tidy_df)


## Plot 4: Question-Level Response-Category Mismatch Rate Heatmaps

This drill-down shows which exact questions carry the divergence signal. Zero-rate cells are muted, and cells at or above 10% are annotated.


In [ ]:
def question_label(question_number):
    return f"Q{int(question_number)}"


def plot_question_scenario_model_side_mismatch_rates(df):
    conditions = condition_values(df)
    families = survey_families(df)
    vmax = mismatch_rate_vmax(df, ["participant_role", "question_family", "question_number", "condition", "scenario", "model"])
    cmap = mismatch_rate_cmap()

    for role in participant_roles(df):
        role_df = df[df["participant_role"] == role]
        for family in families:
            family_df = role_df[role_df["question_family"] == family]
            questions = sorted(family_df["question_number"].dropna().unique())
            if not questions:
                continue

            fig, axes = plt.subplots(
                len(questions),
                len(conditions),
                figsize=(max(10, 5.4 * len(conditions)), max(4, 2.25 * len(questions))),
                squeeze=False,
                sharex=False,
                sharey=False,
                constrained_layout=True,
            )
            im = None
            for r, question_number in enumerate(questions):
                for c, condition in enumerate(conditions):
                    ax = axes[r, c]
                    part = family_df[
                        (family_df["question_number"] == question_number)
                        & (family_df["condition"] == condition)
                    ]
                    table = (
                        part.groupby(["scenario", "model"], observed=True)["public_private_side_mismatch"]
                        .mean()
                        .mul(100)
                        .unstack("model")
                        .sort_index()
                    )
                    table = table.reindex(sorted(table.columns), axis=1).fillna(0)
                    values = table.to_numpy(dtype=float)
                    masked_values = np.ma.masked_where(values <= 0, values)
                    im = ax.imshow(masked_values, aspect="auto", cmap=cmap, vmin=0, vmax=vmax)
                    style_heatmap_axes(ax, table, show_x=(r == len(questions) - 1))
                    annotate_heatmap(ax, values, vmax=vmax)
                    if r == 0:
                        ax.set_title(condition.replace("-", " "), fontsize=11, pad=10)
                    if c == 0:
                        ax.set_ylabel(question_label(question_number), fontsize=10, fontweight="bold")
                    else:
                        ax.set_ylabel("")
            fig.suptitle(
                f"{role.title()} {family} question-level response-category mismatch rate",
                fontsize=14,
                y=1.03,
            )
            colorbar = fig.colorbar(im, ax=axes.ravel().tolist(), label="response-category mismatch rate", fraction=0.025, pad=0.015)
            colorbar.ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0f}%")
            plt.show()

plot_question_scenario_model_side_mismatch_rates(tidy_df)


## Top Question-Level Divergence Slices

This table aggregates over scenario/model so each row has a larger denominator. Use it to identify which survey questions carry the divergence signal.


In [ ]:
top_question_divergence = (
    tidy_df.groupby(["condition", "condition_time", "question_family", "question_number", "participant_role"], observed=True)
    .agg(
        n=("public_private_side_mismatch", "size"),
        side_mismatch_count=("public_private_side_mismatch", "sum"),
        side_mismatch_rate=("public_private_side_mismatch", "mean"),
        large_gap_count=("large_abs_gap", "sum"),
        large_gap_rate=("large_abs_gap", "mean"),
        q95_abs_diff=("abs_public_private_diff", lambda s: s.quantile(0.95)),
        q99_abs_diff=("abs_public_private_diff", lambda s: s.quantile(0.99)),
        max_abs_diff=("abs_public_private_diff", "max"),
    )
    .reset_index()
)
top_question_divergence = (
    top_question_divergence[top_question_divergence["side_mismatch_count"] > 0]
    .sort_values(["side_mismatch_rate", "side_mismatch_count", "q99_abs_diff"], ascending=False)
)
display(top_question_divergence.head(80).round(4))


## Plot 5: Any Public/Private Difference Rate Heatmaps

This is a looser divergence view than response-category mismatch. A cell is high whenever public and private survey scores are not exactly the same, even if they remain on the same side of neutral.


In [ ]:
ANY_DIFFERENCE_EPSILON = 1e-9

tidy_df["public_private_any_difference"] = tidy_df["abs_public_private_diff"] > ANY_DIFFERENCE_EPSILON
print("Any public/private difference count:", int(tidy_df["public_private_any_difference"].sum()))
print("Any public/private difference share:", round(float(tidy_df["public_private_any_difference"].mean()), 4))


def any_difference_rate_vmax(df, group_cols):
    rates = df.groupby(group_cols, observed=True)["public_private_any_difference"].mean().mul(100)
    return max(5.0, float(rates.max()))


def any_difference_cmap():
    cmap = plt.get_cmap("PuBuGn").copy()
    cmap.set_bad("#f4f4f4")
    return cmap


def plot_scenario_model_any_difference_rates(df):
    conditions = condition_values(df)
    families = survey_families(df)
    vmax = any_difference_rate_vmax(df, ["participant_role", "question_family", "condition", "scenario", "model"])
    cmap = any_difference_cmap()

    for role in participant_roles(df):
        role_df = df[df["participant_role"] == role]
        fig, axes = plt.subplots(
            len(families),
            len(conditions),
            figsize=(max(10, 5.4 * len(conditions)), 3.3 * len(families)),
            squeeze=False,
            sharex=False,
            sharey=False,
            constrained_layout=True,
        )
        im = None
        for r, family in enumerate(families):
            for c, condition in enumerate(conditions):
                ax = axes[r, c]
                part = role_df[(role_df["condition"] == condition) & (role_df["question_family"] == family)]
                table = (
                    part.groupby(["scenario", "model"], observed=True)["public_private_any_difference"]
                    .mean()
                    .mul(100)
                    .unstack("model")
                    .sort_index()
                )
                table = table.reindex(sorted(table.columns), axis=1).fillna(0)
                values = table.to_numpy(dtype=float)
                masked_values = np.ma.masked_where(values <= 0, values)
                im = ax.imshow(masked_values, aspect="auto", cmap=cmap, vmin=0, vmax=vmax)
                style_heatmap_axes(ax, table, show_x=(r == len(families) - 1))
                annotate_heatmap(ax, values, vmax=vmax)
                if r == 0:
                    ax.set_title(condition.replace("-", " "), fontsize=11, pad=10)
                if c == 0:
                    ax.set_ylabel(family, fontsize=10, fontweight="bold")
                else:
                    ax.set_ylabel("")
        fig.suptitle(f"{role.title()} public/private any-difference rate", fontsize=14, y=1.04)
        colorbar = fig.colorbar(im, ax=axes.ravel().tolist(), label="any-difference rate", fraction=0.025, pad=0.015)
        colorbar.ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0f}%")
        plt.show()

plot_scenario_model_any_difference_rates(tidy_df)


## Plot 6: Aggregated Model x Condition Heatmaps

These two views collapse over scenarios, survey families, individual questions, and turns. Alpha and beta are kept separate, while future/historical incentive timing is folded into the broader condition label.


In [ ]:
if "public_private_any_difference" not in tidy_df.columns:
    tidy_df["public_private_any_difference"] = tidy_df["abs_public_private_diff"] > ANY_DIFFERENCE_EPSILON


def aggregate_role_condition_model_rate(df, signal_column, role):
    order = [condition for condition in CONDITION_ORDER if condition in set(df["condition"].dropna())]
    role_df = df[df["participant_role"] == role]
    table = (
        role_df.groupby(["condition", "model"], observed=True)[signal_column]
        .mean()
        .mul(100)
        .unstack("model")
    )
    table = table.reindex(order)
    table = table.reindex(sorted(table.columns), axis=1).fillna(0)
    return table


def style_aggregate_heatmap_axes(ax, table, *, show_y=True):
    ax.set_xticks(np.arange(table.shape[1]))
    ax.set_xticklabels([model_label(m) for m in table.columns], rotation=35, ha="right", fontsize=8)
    ax.set_yticks(np.arange(table.shape[0]))
    if show_y:
        ax.set_yticklabels([str(label).replace("-", " ") for label in table.index], fontsize=10)
        ax.set_ylabel("condition", fontsize=10)
    else:
        ax.set_yticklabels([])
        ax.set_ylabel("")
    ax.set_xticks(np.arange(-0.5, table.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, table.shape[0], 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.2)
    ax.tick_params(which="minor", bottom=False, left=False)
    ax.set_xlabel("model", fontsize=10)


def aggregate_rate_vmax(df, signal_column):
    rates = (
        df.groupby(["participant_role", "condition", "model"], observed=True)[signal_column]
        .mean()
        .mul(100)
    )
    return max(5.0, float(rates.max()))


def plot_aggregate_condition_model_heatmaps_by_role(df, signal_column, title, colorbar_label, cmap):
    roles = participant_roles(df)
    vmax = aggregate_rate_vmax(df, signal_column)
    fig, axes = plt.subplots(
        1,
        len(roles),
        figsize=(max(10, 0.72 * df["model"].nunique() * len(roles)), 4.1),
        squeeze=False,
        constrained_layout=True,
    )
    axes = axes[0]
    im = None
    for index, (ax, role) in enumerate(zip(axes, roles)):
        table = aggregate_role_condition_model_rate(df, signal_column, role)
        values = table.to_numpy(dtype=float)
        masked_values = np.ma.masked_where(values <= 0, values)
        im = ax.imshow(masked_values, aspect="auto", cmap=cmap, vmin=0, vmax=vmax)
        style_aggregate_heatmap_axes(ax, table, show_y=(index == 0))
        annotate_heatmap(ax, values, vmax=vmax)
        ax.set_title(role.title(), fontsize=12, pad=10)
    fig.suptitle(title, fontsize=14, y=1.05)
    colorbar = fig.colorbar(im, ax=axes.tolist(), label=colorbar_label, fraction=0.025, pad=0.015)
    colorbar.ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0f}%")
    plt.show()

    for role in roles:
        print(role.title())
        display(aggregate_role_condition_model_rate(df, signal_column, role).round(2))

plot_aggregate_condition_model_heatmaps_by_role(
    tidy_df,
    "public_private_side_mismatch",
    "Aggregated public/private response-category mismatch rate by role",
    "response-category mismatch rate",
    mismatch_rate_cmap(),
)

plot_aggregate_condition_model_heatmaps_by_role(
    tidy_df,
    "public_private_any_difference",
    "Aggregated public/private any-difference rate by role",
    "any-difference rate",
    any_difference_cmap(),
)


## Plot 7: Aggregated Model x Condition Heatmaps Relative To Baseline

These heatmaps use the same aggregation as Plot 6, but each value is shown relative to that model and role's baseline rate. Values are percentage-point changes, not raw rates.


In [ ]:
def aggregate_role_condition_model_delta_from_baseline(df, signal_column, role):
    table = aggregate_role_condition_model_rate(df, signal_column, role)
    if "baseline" not in table.index:
        return table * np.nan
    return table.subtract(table.loc["baseline"], axis=1)


def format_percentage_point_delta(value):
    if not np.isfinite(value):
        return ""
    if abs(value) < 0.05:
        return "0.0"
    return f"{value:+.1f}"


def annotate_delta_heatmap(ax, values, *, vmax):
    contrast_cutoff = max(5.0, 0.55 * vmax)
    for row in range(values.shape[0]):
        for col in range(values.shape[1]):
            value = values[row, col]
            label = format_percentage_point_delta(value)
            if not label:
                continue
            if abs(value) >= contrast_cutoff:
                color = "white"
                weight = "bold"
            elif abs(value) < 0.05:
                color = "#777777"
                weight = "normal"
            else:
                color = "#202020"
                weight = "normal"
            ax.text(col, row, label, ha="center", va="center", fontsize=7.5, fontweight=weight, color=color)


def aggregate_delta_vmax(df, signal_column):
    values = []
    for role in participant_roles(df):
        table = aggregate_role_condition_model_delta_from_baseline(df, signal_column, role)
        values.append(table.to_numpy(dtype=float).ravel())
    all_values = np.concatenate(values) if values else np.array([0.0])
    finite_values = all_values[np.isfinite(all_values)]
    if finite_values.size == 0:
        return 1.0
    return max(1.0, float(np.nanmax(np.abs(finite_values))))


def plot_aggregate_condition_model_delta_heatmaps_by_role(df, signal_column, title, colorbar_label):
    roles = participant_roles(df)
    vmax = aggregate_delta_vmax(df, signal_column)
    cmap = plt.get_cmap("RdBu_r")
    fig, axes = plt.subplots(
        1,
        len(roles),
        figsize=(max(10, 0.72 * df["model"].nunique() * len(roles)), 4.1),
        squeeze=False,
        constrained_layout=True,
    )
    axes = axes[0]
    im = None
    for index, (ax, role) in enumerate(zip(axes, roles)):
        table = aggregate_role_condition_model_delta_from_baseline(df, signal_column, role)
        values = table.to_numpy(dtype=float)
        im = ax.imshow(values, aspect="auto", cmap=cmap, vmin=-vmax, vmax=vmax)
        style_aggregate_heatmap_axes(ax, table, show_y=(index == 0))
        annotate_delta_heatmap(ax, values, vmax=vmax)
        ax.set_title(role.title(), fontsize=12, pad=10)
    fig.suptitle(title, fontsize=14, y=1.05)
    colorbar = fig.colorbar(im, ax=axes.tolist(), label=colorbar_label, fraction=0.025, pad=0.015)
    colorbar.ax.yaxis.set_major_formatter(lambda value, _: f"{value:+.0f} pp")
    plt.show()

    for role in roles:
        print(role.title())
        display(aggregate_role_condition_model_delta_from_baseline(df, signal_column, role).round(2))

plot_aggregate_condition_model_delta_heatmaps_by_role(
    tidy_df,
    "public_private_side_mismatch",
    "Response-category mismatch rate relative to baseline",
    "change from baseline",
)

plot_aggregate_condition_model_delta_heatmaps_by_role(
    tidy_df,
    "public_private_any_difference",
    "Any-difference rate relative to baseline",
    "change from baseline",
)


## Plot 8: Any-Difference Rate Distributions Over Scenario/Question Slices

The aggregate heatmap gives one mean rate per model and condition. This plot keeps the collapsed dimensions visible by first computing any-difference rates for each `scenario x survey family x question` slice, then showing the distribution of those slice-level rates.


In [ ]:
if "public_private_any_difference" not in tidy_df.columns:
    tidy_df["public_private_any_difference"] = tidy_df["abs_public_private_diff"] > ANY_DIFFERENCE_EPSILON


def any_difference_slice_rates(df):
    return (
        df.groupby(
            ["participant_role", "condition", "model", "scenario", "question_family", "question_number"],
            observed=True,
        )["public_private_any_difference"]
        .mean()
        .mul(100)
        .reset_index(name="any_difference_rate")
    )


def condition_box_colors(conditions):
    return [CONDITION_COLORS.get(condition, "#999999") for condition in conditions]


def plot_any_difference_rate_distributions(df):
    slice_df = any_difference_slice_rates(df)
    conditions = [condition for condition in CONDITION_ORDER if condition in set(slice_df["condition"])]
    models = sorted(slice_df["model"].dropna().unique())
    width = 0.22
    offsets = np.linspace(-width, width, len(conditions)) if len(conditions) > 1 else np.array([0.0])

    for role in participant_roles(df):
        role_df = slice_df[slice_df["participant_role"] == role]
        fig, ax = plt.subplots(figsize=(max(11, 0.82 * len(models)), 5.8), constrained_layout=True)
        legend_handles = []
        for condition, offset, color in zip(conditions, offsets, condition_box_colors(conditions)):
            data = [
                role_df[(role_df["model"] == model) & (role_df["condition"] == condition)]["any_difference_rate"].to_numpy(dtype=float)
                for model in models
            ]
            positions = np.arange(len(models)) + offset
            box = ax.boxplot(
                data,
                positions=positions,
                widths=width * 0.85,
                patch_artist=True,
                manage_ticks=False,
                showfliers=True,
                flierprops={"marker": ".", "markersize": 3, "markerfacecolor": color, "markeredgecolor": color, "alpha": 0.45},
                medianprops={"color": "#202020", "linewidth": 1.4},
                whiskerprops={"color": "#666666", "linewidth": 1},
                capprops={"color": "#666666", "linewidth": 1},
            )
            for patch in box["boxes"]:
                patch.set_facecolor(color)
                patch.set_alpha(0.35)
                patch.set_edgecolor("#444444")
            legend_handles.append(plt.Line2D([0], [0], color=color, linewidth=8, alpha=0.5, label=condition.replace("-", " ")))
        ax.set_xticks(np.arange(len(models)))
        ax.set_xticklabels([model_label(model) for model in models], rotation=35, ha="right", fontsize=8)
        ax.set_ylabel("any-difference rate per scenario/question slice (%)")
        ax.set_xlabel("model")
        ax.set_ylim(-2, 102)
        ax.set_title(f"{role.title()} distribution of public/private any-difference rates", fontsize=14, pad=12)
        ax.legend(handles=legend_handles, frameon=False, ncols=len(conditions), loc="upper left", bbox_to_anchor=(0, 1.08))
        ax.grid(axis="y", color="#dddddd", linewidth=0.8)
        plt.show()

plot_any_difference_rate_distributions(tidy_df)


## Plot 9: Any-Difference Rate Distributions Relative To Baseline

This uses the same `scenario x survey family x question` slice rates as Plot 8, but subtracts the matching baseline slice for the same role, model, scenario, family, and question. Values are percentage-point changes from baseline.


In [ ]:
def any_difference_slice_deltas_from_baseline(df):
    slice_df = any_difference_slice_rates(df)
    index_cols = ["participant_role", "model", "scenario", "question_family", "question_number"]
    wide = slice_df.pivot_table(
        index=index_cols,
        columns="condition",
        values="any_difference_rate",
        aggfunc="mean",
        observed=True,
    )
    if "baseline" not in wide.columns:
        return pd.DataFrame(columns=index_cols + ["condition", "delta_from_baseline"])
    delta_cols = [condition for condition in CONDITION_ORDER if condition in wide.columns and condition != "baseline"]
    deltas = wide[delta_cols].subtract(wide["baseline"], axis=0).reset_index()
    return deltas.melt(
        id_vars=index_cols,
        value_vars=delta_cols,
        var_name="condition",
        value_name="delta_from_baseline",
    ).dropna(subset=["delta_from_baseline"])


def plot_any_difference_delta_distributions(df):
    delta_df = any_difference_slice_deltas_from_baseline(df)
    conditions = [condition for condition in CONDITION_ORDER if condition in set(delta_df["condition"])]
    models = sorted(delta_df["model"].dropna().unique())
    width = 0.26
    offsets = np.linspace(-width / 2, width / 2, len(conditions)) if len(conditions) > 1 else np.array([0.0])
    max_abs = max(5.0, float(np.nanpercentile(np.abs(delta_df["delta_from_baseline"]), 99))) if not delta_df.empty else 5.0
    ylimit = min(100.0, max_abs * 1.2)

    for role in participant_roles(df):
        role_df = delta_df[delta_df["participant_role"] == role]
        fig, ax = plt.subplots(figsize=(max(11, 0.82 * len(models)), 5.8), constrained_layout=True)
        legend_handles = []
        for condition, offset, color in zip(conditions, offsets, condition_box_colors(conditions)):
            data = [
                role_df[(role_df["model"] == model) & (role_df["condition"] == condition)]["delta_from_baseline"].to_numpy(dtype=float)
                for model in models
            ]
            positions = np.arange(len(models)) + offset
            box = ax.boxplot(
                data,
                positions=positions,
                widths=width * 0.85,
                patch_artist=True,
                manage_ticks=False,
                showfliers=True,
                flierprops={"marker": ".", "markersize": 3, "markerfacecolor": color, "markeredgecolor": color, "alpha": 0.45},
                medianprops={"color": "#202020", "linewidth": 1.4},
                whiskerprops={"color": "#666666", "linewidth": 1},
                capprops={"color": "#666666", "linewidth": 1},
            )
            for patch in box["boxes"]:
                patch.set_facecolor(color)
                patch.set_alpha(0.35)
                patch.set_edgecolor("#444444")
            legend_handles.append(plt.Line2D([0], [0], color=color, linewidth=8, alpha=0.5, label=condition.replace("-", " ")))
        ax.axhline(0, color="#222222", linewidth=1)
        ax.set_xticks(np.arange(len(models)))
        ax.set_xticklabels([model_label(model) for model in models], rotation=35, ha="right", fontsize=8)
        ax.set_ylabel("change from baseline per scenario/question slice (pp)")
        ax.set_xlabel("model")
        ax.set_ylim(-ylimit, ylimit)
        ax.set_title(f"{role.title()} distribution of any-difference rate changes from baseline", fontsize=14, pad=12)
        ax.legend(handles=legend_handles, frameon=False, ncols=max(1, len(conditions)), loc="upper left", bbox_to_anchor=(0, 1.08))
        ax.grid(axis="y", color="#dddddd", linewidth=0.8)
        plt.show()

plot_any_difference_delta_distributions(tidy_df)


## Plot 10: Response-Category Mismatch Rate Distributions Over Scenario/Question Slices

This is the response-category mismatch counterpart to Plot 8. It computes mismatch rates for each `scenario x survey family x question` slice first, then shows the distribution of those slice-level rates. Because this signal is sparse and zero-heavy, the default view is a violin with a compact inner box; change `CATEGORY_MISMATCH_DISTRIBUTION_KIND` to `"box"` if you want classic boxplots only.


In [ ]:
CATEGORY_MISMATCH_DISTRIBUTION_KIND = "violin"  # options: "violin", "box"
SHOW_MISMATCH_DISTRIBUTION_POINTS = True


def category_mismatch_slice_rates(df):
    return (
        df.groupby(
            ["participant_role", "condition", "model", "scenario", "question_family", "question_number"],
            observed=True,
        )["public_private_side_mismatch"]
        .mean()
        .mul(100)
        .reset_index(name="category_mismatch_rate")
    )


def clean_distribution_values(values):
    values = np.asarray(values, dtype=float)
    return values[np.isfinite(values)]


def draw_grouped_distribution(ax, data, positions, width, color, *, kind):
    clean_data = [clean_distribution_values(values) for values in data]
    non_empty = [(pos, values) for pos, values in zip(positions, clean_data) if values.size]
    if not non_empty:
        return

    non_empty_positions = [pos for pos, _ in non_empty]
    non_empty_data = [values for _, values in non_empty]
    if kind == "violin":
        variable_data = []
        variable_positions = []
        for pos, values in non_empty:
            if np.nanmax(values) > np.nanmin(values):
                variable_positions.append(pos)
                variable_data.append(values)
            else:
                ax.hlines(values[0], pos - width * 0.34, pos + width * 0.34, color=color, linewidth=8, alpha=0.28, zorder=1)
        if variable_data:
            violin = ax.violinplot(
                variable_data,
                positions=variable_positions,
                widths=width * 0.95,
                showmeans=False,
                showmedians=False,
                showextrema=False,
            )
            for body in violin["bodies"]:
                body.set_facecolor(color)
                body.set_edgecolor(color)
                body.set_alpha(0.26)
                body.set_linewidth(0.8)
                body.set_zorder(1)
        box_width = width * 0.34
    elif kind == "box":
        box_width = width * 0.72
    else:
        raise ValueError(f"Unknown distribution kind: {kind!r}")

    box = ax.boxplot(
        non_empty_data,
        positions=non_empty_positions,
        widths=box_width,
        patch_artist=True,
        manage_ticks=False,
        showfliers=False,
        medianprops={"color": "#111111", "linewidth": 1.7},
        whiskerprops={"color": "#444444", "linewidth": 1.05},
        capprops={"color": "#444444", "linewidth": 1.05},
    )
    for patch in box["boxes"]:
        patch.set_facecolor("white")
        patch.set_alpha(0.82)
        patch.set_edgecolor("#222222")
        patch.set_linewidth(1.15)
        patch.set_zorder(3)

    if SHOW_MISMATCH_DISTRIBUTION_POINTS:
        rng = np.random.default_rng(7)
        jitter_scale = width * (0.13 if kind == "violin" else 0.22)
        for pos, values in non_empty:
            jitter = rng.normal(0, jitter_scale, size=values.size)
            ax.scatter(
                np.full(values.size, pos) + jitter,
                values,
                color=color,
                s=8,
                alpha=0.14,
                edgecolors="none",
                zorder=2,
            )


def plot_category_mismatch_rate_distributions(df):
    slice_df = category_mismatch_slice_rates(df)
    conditions = [condition for condition in CONDITION_ORDER if condition in set(slice_df["condition"])]
    models = sorted(slice_df["model"].dropna().unique())
    width = 0.24
    offsets = np.linspace(-width, width, len(conditions)) if len(conditions) > 1 else np.array([0.0])

    for role in participant_roles(df):
        role_df = slice_df[slice_df["participant_role"] == role]
        fig, ax = plt.subplots(figsize=(max(11, 0.82 * len(models)), 5.8), constrained_layout=True)
        legend_handles = []
        for condition, offset, color in zip(conditions, offsets, condition_box_colors(conditions)):
            data = [
                role_df[(role_df["model"] == model) & (role_df["condition"] == condition)]["category_mismatch_rate"].to_numpy(dtype=float)
                for model in models
            ]
            positions = np.arange(len(models)) + offset
            draw_grouped_distribution(ax, data, positions, width, color, kind=CATEGORY_MISMATCH_DISTRIBUTION_KIND)
            legend_handles.append(plt.Line2D([0], [0], color=color, linewidth=8, alpha=0.45, label=condition.replace("-", " ")))
        ax.set_xticks(np.arange(len(models)))
        ax.set_xticklabels([model_label(model) for model in models], rotation=35, ha="right", fontsize=8)
        ax.set_ylabel("response-category mismatch rate per scenario/question slice (%)")
        ax.set_xlabel("model")
        ax.set_ylim(-2, 102)
        ax.set_title(f"{role.title()} distribution of response-category mismatch rates", fontsize=14, pad=12)
        ax.legend(handles=legend_handles, frameon=False, ncols=len(conditions), loc="upper left", bbox_to_anchor=(0, 1.08))
        ax.grid(axis="y", color="#dddddd", linewidth=0.8)
        plt.show()

plot_category_mismatch_rate_distributions(tidy_df)


## Plot 11: Response-Category Mismatch Rate Distributions Relative To Baseline

This uses the same `scenario x survey family x question` slice rates as Plot 10, but subtracts the matching baseline slice for the same role, model, scenario, family, and question. Values are percentage-point changes from baseline. It uses the same configurable violin/box renderer as Plot 10.


In [ ]:
def category_mismatch_slice_deltas_from_baseline(df):
    slice_df = category_mismatch_slice_rates(df)
    index_cols = ["participant_role", "model", "scenario", "question_family", "question_number"]
    wide = slice_df.pivot_table(
        index=index_cols,
        columns="condition",
        values="category_mismatch_rate",
        aggfunc="mean",
        observed=True,
    )
    if "baseline" not in wide.columns:
        return pd.DataFrame(columns=index_cols + ["condition", "delta_from_baseline"])
    delta_cols = [condition for condition in CONDITION_ORDER if condition in wide.columns and condition != "baseline"]
    deltas = wide[delta_cols].subtract(wide["baseline"], axis=0).reset_index()
    return deltas.melt(
        id_vars=index_cols,
        value_vars=delta_cols,
        var_name="condition",
        value_name="delta_from_baseline",
    ).dropna(subset=["delta_from_baseline"])


def plot_category_mismatch_delta_distributions(df):
    delta_df = category_mismatch_slice_deltas_from_baseline(df)
    conditions = [condition for condition in CONDITION_ORDER if condition in set(delta_df["condition"])]
    models = sorted(delta_df["model"].dropna().unique())
    width = 0.26
    offsets = np.linspace(-width / 2, width / 2, len(conditions)) if len(conditions) > 1 else np.array([0.0])
    max_abs = max(5.0, float(np.nanpercentile(np.abs(delta_df["delta_from_baseline"]), 99))) if not delta_df.empty else 5.0
    ylimit = min(100.0, max_abs * 1.2)

    for role in participant_roles(df):
        role_df = delta_df[delta_df["participant_role"] == role]
        fig, ax = plt.subplots(figsize=(max(11, 0.82 * len(models)), 5.8), constrained_layout=True)
        legend_handles = []
        for condition, offset, color in zip(conditions, offsets, condition_box_colors(conditions)):
            data = [
                role_df[(role_df["model"] == model) & (role_df["condition"] == condition)]["delta_from_baseline"].to_numpy(dtype=float)
                for model in models
            ]
            positions = np.arange(len(models)) + offset
            draw_grouped_distribution(ax, data, positions, width, color, kind=CATEGORY_MISMATCH_DISTRIBUTION_KIND)
            legend_handles.append(plt.Line2D([0], [0], color=color, linewidth=8, alpha=0.45, label=condition.replace("-", " ")))
        ax.axhline(0, color="#222222", linewidth=1)
        ax.set_xticks(np.arange(len(models)))
        ax.set_xticklabels([model_label(model) for model in models], rotation=35, ha="right", fontsize=8)
        ax.set_ylabel("response-category mismatch change from baseline per slice (pp)")
        ax.set_xlabel("model")
        ax.set_ylim(-ylimit, ylimit)
        ax.set_title(f"{role.title()} distribution of response-category mismatch changes from baseline", fontsize=14, pad=12)
        ax.legend(handles=legend_handles, frameon=False, ncols=max(1, len(conditions)), loc="upper left", bbox_to_anchor=(0, 1.08))
        ax.grid(axis="y", color="#dddddd", linewidth=0.8)
        plt.show()

plot_category_mismatch_delta_distributions(tidy_df)
